# Derivations for Numerical Solution of Geodesics for the Schwarzschild Metric

In [58]:
# Load sympy
from sympy import *

# Setup symbolic variables
t, r, theta, phi, GM = symbols('t r theta phi GM', real=True)

Create and fill the coordinates $x^\mu$, the Schwarzschild Metric Tensor $g_{\mu \nu}$, and its inverse $g^{\mu \nu}$

Spherical coordinates are used: $x^\mu = \begin{pmatrix} t \\ r \\ \theta \\ \phi \end{pmatrix}$

In [59]:
x = MutableDenseNDimArray.zeros(4)
x[0] = t
x[1] = r
x[2] = theta
x[3] = phi

x

[t, r, theta, phi]

In [60]:
g = MutableDenseNDimArray.zeros(4, 4)
g[0, 0] = - (1 - 2*GM/r)
g[1, 1] = 1 / (1 - 2*GM/r)
g[2, 2] = r**2
g[3, 3] = r**2 * sin(theta)**2

g

[[2*GM/r - 1, 0, 0, 0], [0, 1/(-2*GM/r + 1), 0, 0], [0, 0, r**2, 0], [0, 0, 0, r**2*sin(theta)**2]]

In [61]:
g_inv = MutableDenseNDimArray.zeros(4, 4)
for mu in range(4):
    g_inv[mu, mu] = 1 / g[mu, mu]

g_inv

[[1/(2*GM/r - 1), 0, 0, 0], [0, -2*GM/r + 1, 0, 0], [0, 0, r**(-2), 0], [0, 0, 0, 1/(r**2*sin(theta)**2)]]

Generate the Christoffel Symbols $\Gamma^\lambda_{\mu\nu}$

In [62]:
christ_symbs = MutableDenseNDimArray.zeros(4, 4, 4)

# The three indexes of the Christoffel symbols
for lamb in range(4):
    for mu in range(4):
        for nu in range(4):

            # The dummy index for the summation
            for sigma in range(4):
                christ_symbs[lamb, mu, nu] += Rational(1, 2) * g_inv[lamb, sigma] * \
                    (diff(g[nu, sigma], x[mu]) + diff(g[sigma, mu], x[nu]) - diff(g[mu, nu], x[sigma]))

christ_symbs

[[[0, -GM/(r**2*(2*GM/r - 1)), 0, 0], [-GM/(r**2*(2*GM/r - 1)), 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]], [[2*GM*(-GM/r + 1/2)/r**2, 0, 0, 0], [0, -2*GM*(-GM/r + 1/2)/(r**2*(-2*GM/r + 1)**2), 0, 0], [0, 0, -2*r*(-GM/r + 1/2), 0], [0, 0, 0, -2*r*(-GM/r + 1/2)*sin(theta)**2]], [[0, 0, 0, 0], [0, 0, 1/r, 0], [0, 1/r, 0, 0], [0, 0, 0, -sin(theta)*cos(theta)]], [[0, 0, 0, 0], [0, 0, 0, 1/r], [0, 0, 0, cos(theta)/sin(theta)], [0, 1/r, cos(theta)/sin(theta), 0]]]

Now generate equations of motion $\frac{\text{d}^2 x^\mu}{\text{d}\tau^2}$

In [63]:
# First define symbols for the derivative of the coordinates
dt, dr, dtheta, dphi = symbols('tdot rdot thetadot phidot', real=True)

dx = MutableDenseNDimArray.zeros(4)
dx[0] = dt
dx[1] = dr
dx[2] = dtheta
dx[3] = dphi

dx

[tdot, rdot, thetadot, phidot]

In [64]:
ddx = MutableDenseNDimArray.zeros(4)

# The index of the second derivative coordinate
for mu in range(4):
    
    # Summations over the chrstoffel symbols and first derivatives
    for rho in range(4):
        for sigma in range(4):
            ddx[mu] -= christ_symbs[mu, rho, sigma] * dx[rho] * dx[sigma]

ddx = simplify(ddx)
ddx

[2*GM*rdot*tdot/(r*(2*GM - r)), (-GM*r**2*rdot**2 + GM*tdot**2*(2*GM - r)**2 - r**3*(2*GM - r)**2*(phidot**2*sin(theta)**2 + thetadot**2))/(r**3*(2*GM - r)), phidot**2*sin(2*theta)/2 - 2*rdot*thetadot/r, -2*phidot*(r*thetadot/tan(theta) + rdot)/r]

Convert the resulting equation to python code

In [65]:
# Print function code
from sympy.printing.numpy import NumPyPrinter
NumPyPrinter().doprint(ddx)

'numpy.array([2*GM*rdot*tdot/(r*(2*GM - r)), (-GM*r**2*rdot**2 + GM*tdot**2*(2*GM - r)**2 - r**3*(2*GM - r)**2*(phidot**2*numpy.sin(theta)**2 + thetadot**2))/(r**3*(2*GM - r)), (1/2)*phidot**2*numpy.sin(2*theta) - 2*rdot*thetadot/r, -2*phidot*(r*thetadot/numpy.tan(theta) + rdot)/r])'

In [66]:
# Print used variables
ddx.free_symbols

{GM, phidot, r, rdot, tdot, theta, thetadot}

Now add some coordinate transformation code

In [67]:
cart = MutableDenseNDimArray.zeros(4)

cart[0] = t
cart[1] = r * sin(theta) * cos(phi)
cart[2] = r * sin(theta) * sin(phi)
cart[3] = r * cos(theta)

cart

[t, r*sin(theta)*cos(phi), r*sin(phi)*sin(theta), r*cos(theta)]

In [68]:
# Create coordinate transformation matrix: spherical to cartesian
dcart_dsph = Matrix.zeros(4, 4)

for mu_ in range(4):
    for mu in range(4):
        dcart_dsph[mu_, mu] = diff(cart[mu_], x[mu])

# Take inverse: cartesian to spherical
dsph_dcart = simplify(dcart_dsph.inv())

In [69]:
dcart_dsph

Matrix([
[1,                   0,                     0,                      0],
[0, sin(theta)*cos(phi), r*cos(phi)*cos(theta), -r*sin(phi)*sin(theta)],
[0, sin(phi)*sin(theta), r*sin(phi)*cos(theta),  r*sin(theta)*cos(phi)],
[0,          cos(theta),         -r*sin(theta),                      0]])

In [70]:
dsph_dcart

Matrix([
[1,                        0,                       0,             0],
[0,      sin(theta)*cos(phi),     sin(phi)*sin(theta),    cos(theta)],
[0,    cos(phi)*cos(theta)/r,   sin(phi)*cos(theta)/r, -sin(theta)/r],
[0, -sin(phi)/(r*sin(theta)), cos(phi)/(r*sin(theta)),             0]])

Spherical velocity to cartesian velocity

In [71]:
dcart = MutableDenseNDimArray.zeros(4)

for mu_ in range(4):
    for mu in range(4):
        dcart[mu_] += dcart_dsph[mu_, mu] * dx[mu]

dcart

[tdot, -phidot*r*sin(phi)*sin(theta) + r*thetadot*cos(phi)*cos(theta) + rdot*sin(theta)*cos(phi), phidot*r*sin(theta)*cos(phi) + r*thetadot*sin(phi)*cos(theta) + rdot*sin(phi)*sin(theta), -r*thetadot*sin(theta) + rdot*cos(theta)]

In [72]:
NumPyPrinter().doprint(dcart)

'numpy.array([tdot, -phidot*r*numpy.sin(phi)*numpy.sin(theta) + r*thetadot*numpy.cos(phi)*numpy.cos(theta) + rdot*numpy.sin(theta)*numpy.cos(phi), phidot*r*numpy.sin(theta)*numpy.cos(phi) + r*thetadot*numpy.sin(phi)*numpy.cos(theta) + rdot*numpy.sin(phi)*numpy.sin(theta), -r*thetadot*numpy.sin(theta) + rdot*numpy.cos(theta)])'

In [73]:
dcart.free_symbols

{phi, phidot, r, rdot, tdot, theta, thetadot}

Cartesian velocity to spherical velocity

In [74]:
dt_cart, dx_cart, dy_cart, dz_cart = symbols('tdot_cart xdot_cart ydot_cart zdot_cart')
dcart_symb = MutableDenseNDimArray.zeros(4)
dcart_symb[0] = dt_cart
dcart_symb[1] = dx_cart
dcart_symb[2] = dy_cart
dcart_symb[3] = dz_cart

dx_conv = MutableDenseNDimArray.zeros(4)

for mu in range(4):
    for mu_ in range(4):
        dx_conv[mu] += dsph_dcart[mu, mu_] * dcart_symb[mu_]

dx_conv = simplify(dx_conv)
dx_conv

[tdot_cart, xdot_cart*sin(theta)*cos(phi) + ydot_cart*sin(phi)*sin(theta) + zdot_cart*cos(theta), (xdot_cart*cos(phi)*cos(theta) + ydot_cart*sin(phi)*cos(theta) - zdot_cart*sin(theta))/r, (-xdot_cart*sin(phi) + ydot_cart*cos(phi))/(r*sin(theta))]

In [75]:
NumPyPrinter().doprint(dx_conv)

'numpy.array([tdot_cart, xdot_cart*numpy.sin(theta)*numpy.cos(phi) + ydot_cart*numpy.sin(phi)*numpy.sin(theta) + zdot_cart*numpy.cos(theta), (xdot_cart*numpy.cos(phi)*numpy.cos(theta) + ydot_cart*numpy.sin(phi)*numpy.cos(theta) - zdot_cart*numpy.sin(theta))/r, (-xdot_cart*numpy.sin(phi) + ydot_cart*numpy.cos(phi))/(r*numpy.sin(theta))])'

In [76]:
dx_conv.free_symbols

{phi, r, tdot_cart, theta, xdot_cart, ydot_cart, zdot_cart}

Make equation to enforce that initial four-velocity satisfies $U_\mu U^\mu = -1$

In [ ]:
# Compute norm
dx_norm = 0

for mu in range(4):
    for nu in range(4):
        dx_norm += g[mu, nu] * dx[nu] * dx[mu]

dx_norm

phidot**2*r**2*sin(theta)**2 + r**2*thetadot**2 + rdot**2/(-2*GM/r + 1) + tdot**2*(2*GM/r - 1)

In [86]:
# Should be the positive of the two solutions
dt_sol = solve(Eq(dx_norm, -1), dt)[0]
dt_sol

sqrt(r*(-2*GM*phidot**2*r**2*sin(theta)**2 - 2*GM*r**2*thetadot**2 - 2*GM + phidot**2*r**3*sin(theta)**2 + r**3*thetadot**2 + r*rdot**2 + r))/(-2*GM + r)

In [89]:
NumPyPrinter().doprint(dt_sol)

'numpy.sqrt(r*(-2*GM*phidot**2*r**2*numpy.sin(theta)**2 - 2*GM*r**2*thetadot**2 - 2*GM + phidot**2*r**3*numpy.sin(theta)**2 + r**3*thetadot**2 + r*rdot**2 + r))/(-2*GM + r)'

In [90]:
dt_sol.free_symbols

{GM, phidot, r, rdot, theta, thetadot}